In [ ]:
from google.colab import drive
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from tqdm.auto import tqdm

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Fitness')   
VIDEO_ROOT = Path('/content/drive/MyDrive/Fitness/Selected_videos')
MANIFEST_CSV = Path('/content/drive/MyDrive/Fitness/online_realtime_105_video_results.csv')
OUT_DIR = Path('/content/drive/MyDrive/Fitness/relative_amplitude_105')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0,str(PROJECT_DIR))
print('Manifest exists:',MANIFEST_CSV.exists(), 'Project exists:',PROJECT_DIR.exists())

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")


example = "-7BiugaROUM_000016_000026.mp4"

matches = list(DRIVE_ROOT.rglob(example))

print("Matching videos found:", len(matches))

for p in matches:
    print(p)

In [ ]:
m = pd.read_csv(MANIFEST_CSV)
assert len(m)==105, f'Expected 105 manifest rows, got {len(m)}'
assert {'video_name','exercise','ground_truth','offline_reps'}.issubset(m.columns)
normalize = {'push_up':'pushup','pushup':'pushup','pushups':'pushup', 'squat':'squat','squats':'squat','situp':'situp','sit_up':'situp','situps':'situp'}
m['exercise_norm']=m.exercise.astype(str).str.lower().map(normalize)
assert m.exercise_norm.notna().all(), 'Unrecognized exercise names'
assert m.video_name.is_unique, 'Duplicate video names in manifest'
assert m.groupby('exercise_norm').size().to_dict()=={'pushup':32,'squat':26,'situp':47}, m.groupby('exercise_norm').size()
print(m.groupby('exercise_norm').agg(videos=('video_name','size'), ground_truth_reps=('ground_truth','sum'), offline_reps=('offline_reps','sum')))


from collections import defaultdict
wanted=set(m.video_name.astype(str))
found=defaultdict(list)
for p in VIDEO_ROOT.rglob('*.mp4'):
    if p.name in wanted: found[p.name].append(p)
missing=sorted(wanted-set(found))
ambiguous={k:[str(x) for x in v] for k,v in found.items() if len(v)>1}
print('Found:',len(found),'Missing:',len(missing),'Ambiguous:',len(ambiguous))
if missing: print('Missing examples:',missing[:10])
if ambiguous: print('Ambiguous examples:',list(ambiguous.items())[:5])
assert not missing and not ambiguous, 'Fix video paths before continuing (do not silently drop or choose duplicates)'
m['video_path']=m.video_name.map(lambda x:str(found[x][0]))

In [ ]:
from movenet import extract_keypoints_from_video
from assessment import (extract_pushup_angles_confidence_aware, extract_squat_angles_confidence_aware, extract_situp_angles)
from util import gaussian_smoothing
PARAMS={'pushup':(0.35,0.75,15.0), 'squat':(0.35,0.75,20.0), 'situp':(0.30,0.70,12.0)}
TARGETS=[0.70,0.80,0.90]

def replay_original_counter(angles, exercise):
    a=np.asarray(angles,dtype=float).reshape(-1)
    if len(a)==0 or not np.isfinite(a).all():
        return [], {'status':'unreliable_angles','count':0}
    lo,hi=float(a.min()),float(a.max()); span=hi-lo
    alpha,beta,minrange=PARAMS[exercise]
    if span < minrange:
        return [], {'status':'insufficient_range','count':0,'theta_min':lo,'theta_max':hi,'range_deg':span}
    down=lo+alpha*span; up=lo+beta*span
    state='UP'; since=999; bottom=float('inf'); start=0; cycles=[]
    for i,theta in enumerate(a):
        since+=1
        if state=='UP':
            if theta<down:
                state='DOWN';bottom=theta
        else:
            bottom=min(bottom,theta)
            if theta>up and since>=8:
                
                if bottom<down and theta>up:
                    seg=a[start:i+1]
                    cycles.append({'rep_index':len(cycles)+1,'start_sample':start,'end_sample':i,
                                   'cycle_min_deg':float(seg.min()),'cycle_max_deg':float(seg.max()),
                                   'bottom_deg':float(bottom),'return_deg':float(theta)})
                state='UP';bottom=float('inf');since=0;start=i
    return cycles, {'status':'ok','count':len(cycles),'theta_min':lo,'theta_max':hi,'range_deg':span,
                    'down_threshold':down,'up_threshold':up,'unclosed_down_phase':int(state=='DOWN')}

def amplitude_flags(cycle, meta):
    
    
    span=meta['range_deg']
    amp=cycle['cycle_max_deg']-cycle['cycle_min_deg']
    result={'cycle_amplitude_deg':amp,'relative_amplitude':amp/span if span>0 else np.nan}
    for t in TARGETS:
        result[f'meets_{int(t*100)}pct']=bool(amp >= t*span)
    return result


synthetic=[170]*8+list(np.linspace(170,70,20))+list(np.linspace(70,170,20))+[170]*8
cy,meta=replay_original_counter(synthetic,'pushup')
assert meta['count']==1,(cy,meta)
print('Synthetic counter smoke test: OK')

In [ ]:
rows=[]; rep_rows=[]; trajectories={}
for rec in tqdm(m.itertuples(index=False), total=len(m)):
    base={'video_name':rec.video_name,'exercise':rec.exercise_norm,
          'ground_truth':int(rec.ground_truth),'reference_offline_reps':int(rec.offline_reps)}
    try:
        kps=extract_keypoints_from_video(rec.video_path)
        if rec.exercise_norm=='pushup': raw=extract_pushup_angles_confidence_aware(kps)
        elif rec.exercise_norm=='squat': raw=extract_squat_angles_confidence_aware(kps)
        else: raw=extract_situp_angles(kps)[0]
        a=np.asarray(gaussian_smoothing(raw),dtype=float).reshape(-1) if len(raw) else np.array([])
        cycles,meta=replay_original_counter(a,rec.exercise_norm)
        base.update(meta);base['angle_samples']=len(a)
        base['matches_reference_offline']=(meta['count']==int(rec.offline_reps))
        if len(a):trajectories[rec.video_name]=a
        for c in cycles:
            rep_rows.append({**{k:base[k] for k in ['video_name','exercise','ground_truth','reference_offline_reps']},
                             **c,**amplitude_flags(c,meta), 'theta_min':meta['theta_min'],
                             'theta_max':meta['theta_max'], 'range_deg':meta['range_deg']})
    except Exception as exc:
        base.update({'status':'error','count':np.nan,'error':repr(exc),'matches_reference_offline':False})
    rows.append(base)
video_df=pd.DataFrame(rows);rep_df=pd.DataFrame(rep_rows)
video_df.to_csv(OUT_DIR/'per_video_105.csv',index=False)
rep_df.to_csv(OUT_DIR/'per_repetition_amplitude_105.csv',index=False)
print('Videos:',len(video_df),'Rep rows:',len(rep_df))
print('Status:',video_df.status.value_counts(dropna=False).to_dict())
print('Offline matches:',int(video_df.matches_reference_offline.sum()),'/',len(video_df))
discrepancies=video_df[~video_df.matches_reference_offline]
discrepancies.to_csv(OUT_DIR/'counter_discrepancies.csv',index=False)
display(discrepancies[['video_name','exercise','reference_offline_reps','count','status']].head(25))
if len(discrepancies):print('WARNING: investigate discrepancies before using amplitude results in manuscript.')

In [ ]:

FORM_DIR = Path('/content/drive/MyDrive/Fitness/Form_Assessment')
FORM_DIR.mkdir(parents=True, exist_ok=True)
meta = video_df.set_index('video_name')
trajectory_rows = []
for name, angles in trajectories.items():
    v = meta.loc[name]
    for i, angle in enumerate(angles):
        trajectory_rows.append({
            'video_name': name, 'exercise': v.exercise,
            'retained_sample_index': i,
            'smoothed_angle_deg': float(angle),
            'down_threshold_deg': v.down_threshold,
            'up_threshold_deg': v.up_threshold})
columns = ['video_name','exercise','retained_sample_index','smoothed_angle_deg','down_threshold_deg','up_threshold_deg']
full_df = pd.DataFrame(trajectory_rows, columns=columns)
full_df.to_csv(FORM_DIR / 'full_angle_trajectories_105.csv', index=False)
video_df.to_csv(FORM_DIR / 'trajectory_export_audit.csv', index=False)
rep_df.to_csv(FORM_DIR / 'per_repetition_amplitude_105_recomputed.csv', index=False)
print('Videos with trajectories:', full_df.video_name.nunique(), '/ 105')
print('Total angle samples:', len(full_df))
print('Detected cycles:', len(rep_df), '(reference: 252)')
print('Count matches:', int(video_df.matches_reference_offline.sum()), '/ 105')
missing = sorted(set(m.video_name) - set(full_df.video_name))
if missing: print('Missing trajectories:', missing)
if missing or len(rep_df) != 252 or not video_df.matches_reference_offline.all():
    print('WARNING: review the audit before treating the trajectories as identical to the original results.')
print('Saved in:', FORM_DIR)
